In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append("../../../../")
from ADFWI.propagator  import *
from ADFWI.model       import *
from ADFWI.view        import *
from ADFWI.utils       import *
from ADFWI.survey      import *
from ADFWI.utils.assessment_metric import *

project_path = "./examples/dip/DIP-ADFWI/01_Multi-CNN/data"

init_model = np.load(os.path.join(project_path,"model/init_model.npz"))
true_model = np.load(os.path.join(project_path,"model/true_model.npz"))
init_v = init_model["vp"]
true_v = true_model["vp"]

ox, oz = 0, 0        
nz, nx = 88, 200      
dx, dz = 40, 40         
nt, dt = 1600, 0.003     
nabc = 30                 
x = np.arange(nx)*dx/1000
z = np.arange(nz)*dz/1000
x_mesh,z_mesh = np.meshgrid(x,z)
src_z = np.array([1  for i in range(2,nx-1,5)])*dz/1000
src_x = np.array([i  for i in range(2,nx-1,5)])*dx/1000
rcv_z = np.array([1  for i in range(0,nx,1)])*dz/1000
rcv_x = np.array([j  for j in range(0,nx,1)])*dz/1000
vmin = true_v.min();vmax = true_v.max() 

In [2]:
iter_vp_no_regular = np.load(os.path.join(project_path,"inversion-no_regularization/iter_vp.npz"))["data"]

iter_vp_2 = np.load(os.path.join(project_path,"inversion-2layer-4-32/iter_vp.npz"))["data"]
iter_vp_3 = np.load(os.path.join(project_path,"inversion-3layer-16-32-16/iter_vp.npz"))["data"]
iter_vp_4 = np.load(os.path.join(project_path,"inversion-4layer-64-32-16-16/iter_vp.npz"))["data"]
iter_vp_5 = np.load(os.path.join(project_path,"inversion-5layer-256-32-32-32-16/iter_vp.npz"))["data"]


In [ ]:
def plot_vel_single_for_all(fig,ax,v,title="",MSE=""):
    plt.rc('font',family='Helvetica')
    # cmap = "bwr"
    cmap = "rainbow"
    # plm = ax.pcolormesh(x_mesh, z_mesh, v,cmap=cmap,vmin=vmin,vmax=vmax)
    x = np.arange(nx*3)*dx/3/1000
    z = np.arange(nz*3)*dz/3/1000
    x_mesh_new,z_mesh_new = np.meshgrid(x,z)
    from scipy.interpolate import griddata
    v_new = griddata((x_mesh.flatten(), z_mesh.flatten()), v.flatten(), (x_mesh_new, z_mesh_new), method='cubic')
    
    plm = ax.pcolormesh(x_mesh_new, z_mesh_new, v_new,cmap=cmap,vmin=vmin,vmax=vmax,shading="nearest")
    ax.invert_yaxis()
    ax.tick_params(labelsize = 16)
    ax.set_title(title,fontsize=16)
    ax.text(5.3,0.35,MSE,fontsize=16,c="w")
    return plm

import matplotlib.transforms as mtransforms
def add_right_cax(ax, pad, width):
    axpos = ax.get_position()
    caxpos = mtransforms.Bbox.from_extents(
        axpos.x1 + pad,
        axpos.y0,
        axpos.x1 + pad + width,
        axpos.y1
    )
    cax = ax.figure.add_axes(caxpos)

    return cax

def plot_vel_singleline_for_all(ax,v_true,v_init,v_inv,x_distance,title,show_xlabel=True,show_ylabel=False,show_legend=False):
    ax.plot(v_true[:,int(x_distance//dx)]/1000,  z, c='k',   linewidth=2, linestyle="-" ,label="True")
    ax.plot(v_init[:,int(x_distance//dx)]/1000,  z, c='gray',linewidth=2, linestyle="-" ,label="Init")
    ax.plot(v_inv [:,int(x_distance//dx)]/1000,  z, c='r',   linewidth=2, linestyle="--",label="Inverted")
    ax.tick_params(labelsize = 16)
    if not show_xlabel:
        ax.set_xticks([])
    # else:
    #     ax.tick_params(labelsize = 15)
        
    if not show_ylabel:
        ax.set_yticks([])
    else:
        ax.tick_params(labelsize = 12)
    ax.invert_yaxis()
    
    if show_legend:
        ax.legend(fontsize = 12)
    ax.set_title(title,fontsize=16)

In [ ]:

fig,axs = plt.subplots(3,2,figsize=(12,8))
im = plot_vel_single_for_all(fig,axs[0][0],true_v    ,title="True Model")
axs[0][0].scatter(rcv_x[1::2],rcv_z[1::2]+0.05,c="w",marker="v",s=10)
axs[0][0].scatter(src_x[1::2],src_z[1::2]+0.05,c="r",marker="*",s=60)
axs[0][0].set_ylabel("Depth (km)",fontsize=15)
axs[0][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[0][1],iter_vp_no_regular[-1],title="No regular"    ,MSE="SSIM:{:.2f}".format(SSIM(true_v[10::],iter_vp_no_regular[-1][10::],win_size=3)))

plot_vel_single_for_all(fig,axs[1][0],iter_vp_2[-1]        ,title=r"3 layer Unet" ,MSE="SSIM:{:.2f}".format(SSIM(true_v[10::],iter_vp_2[-1][10::],win_size=3)))
axs[1][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[1][1],iter_vp_3[-1]        ,title=r"4 layer Unet" ,MSE="SSIM:{:.2f}".format(SSIM(true_v[10::],iter_vp_3[-1][10::],win_size=3)))

plot_vel_single_for_all(fig,axs[2][0],iter_vp_4[-1]        ,title=r"5 layer Unet" ,MSE="SSIM:{:.2f}".format(SSIM(true_v[10::],iter_vp_4[-1][10::],win_size=3)))
axs[2][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[2][1],iter_vp_5[-1]        ,title=r"6 layer Unet" ,MSE="SSIM:{:.2f}".format(SSIM(true_v[10::],iter_vp_5[-1][10::],win_size=3)))


axs[0][0].set_xticks([])
axs[0][1].set_xticks([])
axs[1][0].set_xticks([])
axs[1][1].set_xticks([])
axs[0][1].set_yticks([])
axs[1][1].set_yticks([])
axs[2][1].set_yticks([])

axs[2][0].set_xlabel("Distance (km)", fontsize=15)
axs[2][1].set_xlabel("Distance (km)", fontsize=15)

plt.subplots_adjust(hspace=0.2,wspace=0.1)

cbar = fig.colorbar(im, ax=axs,orientation='vertical',pad = 0.02,shrink=0.6)
cbar.ax.set_title("(m/s)",fontsize=15,loc='center')
cbar.ax.tick_params(labelsize=15)

plt.savefig(os.path.join(project_path,"DIP_CNN.png"),bbox_inches='tight',dpi=300)
plt.show()

## GIF

In [ ]:
###########################################
# visualize the inversion results
###########################################
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

iter_vp = np.load(os.path.join(project_path,"inversion-no_regularization/iter_vp.npz"))["data"][::3]

# Set up the figure for plotting
fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.imshow(iter_vp[0], aspect='equal', cmap='jet_r', vmin=1500, vmax=4146.96)
ax.set_title('Inversion Process Visualization')
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Z Coordinate')
# Create a horizontal colorbar
cbar = fig.colorbar(cax, ax=ax, orientation='horizontal', fraction=0.046, pad=0.2)
cbar.set_label('Velocity (m/s)')
# Adjust the layout to minimize white space
plt.subplots_adjust(top=0.85, bottom=0.2, left=0.1, right=0.9)
# Initialization function
def init():
    cax.set_array(iter_vp[0])  # Use the 2D array directly
    return cax,
# Animation function
def animate(i):
    cax.set_array(iter_vp[i])  # Update with the i-th iteration directly
    return cax,
# Create the animation
ani = animation.FuncAnimation(fig, animate, init_func=init, frames=len(iter_vp), interval=200, blit=True)
# Save the animation as a video file (e.g., MP4 format)
ani.save(os.path.join(project_path,f"inversion-no_regularization/inversion_process.gif"), writer='pillow', fps=10)
# Display the animation using HTML
plt.close(fig)  # Prevents static display of the last frame
HTML(ani.to_jshtml())

## 2-Layer CNN

In [ ]:
###########################################
# visualize the inversion results
###########################################
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

iter_vp = np.load(os.path.join(project_path,"inversion-2layer-4-32/iter_vp.npz"))["data"][::3]
# Set up the figure for plotting
fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.imshow(iter_vp[0], aspect='equal', cmap='jet_r', vmin=1500, vmax=4146.96)
ax.set_title('Inversion Process Visualization')
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Z Coordinate')
# Create a horizontal colorbar
cbar = fig.colorbar(cax, ax=ax, orientation='horizontal', fraction=0.046, pad=0.2)
cbar.set_label('Velocity (m/s)')
# Adjust the layout to minimize white space
plt.subplots_adjust(top=0.85, bottom=0.2, left=0.1, right=0.9)
# Initialization function
def init():
    cax.set_array(iter_vp[0])  # Use the 2D array directly
    return cax,
# Animation function
def animate(i):
    cax.set_array(iter_vp[i])  # Update with the i-th iteration directly
    return cax,
# Create the animation
ani = animation.FuncAnimation(fig, animate, init_func=init, frames=len(iter_vp), interval=200, blit=True)
# Save the animation as a video file (e.g., MP4 format)
ani.save(os.path.join(project_path,f"inversion-2layer-4-32/inversion_process.gif"), writer='pillow', fps=10)
# Display the animation using HTML
plt.close(fig)  # Prevents static display of the last frame
HTML(ani.to_jshtml())

## 3 Layer CNN

In [ ]:
###########################################
# visualize the inversion results
###########################################
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

iter_vp = np.load(os.path.join(project_path,"inversion-3layer-16-32-16/iter_vp.npz"))["data"][::3]

# Set up the figure for plotting
fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.imshow(iter_vp[0], aspect='equal', cmap='jet_r', vmin=1500, vmax=4146.96)
ax.set_title('Inversion Process Visualization')
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Z Coordinate')
# Create a horizontal colorbar
cbar = fig.colorbar(cax, ax=ax, orientation='horizontal', fraction=0.046, pad=0.2)
cbar.set_label('Velocity (m/s)')
# Adjust the layout to minimize white space
plt.subplots_adjust(top=0.85, bottom=0.2, left=0.1, right=0.9)
# Initialization function
def init():
    cax.set_array(iter_vp[0])  # Use the 2D array directly
    return cax,
# Animation function
def animate(i):
    cax.set_array(iter_vp[i])  # Update with the i-th iteration directly
    return cax,
# Create the animation
ani = animation.FuncAnimation(fig, animate, init_func=init, frames=len(iter_vp), interval=200, blit=True)
# Save the animation as a video file (e.g., MP4 format)
ani.save(os.path.join(project_path,f"inversion-3layer-16-32-16/inversion_process.gif"), writer='pillow', fps=10)
# Display the animation using HTML
plt.close(fig)  # Prevents static display of the last frame
HTML(ani.to_jshtml())

## 4 Layer CNN

In [ ]:
###########################################
# visualize the inversion results
###########################################
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

iter_vp = np.load(os.path.join(project_path,"inversion-4layer-64-32-16-16/iter_vp.npz"))["data"][::3]

# Set up the figure for plotting
fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.imshow(iter_vp[0], aspect='equal', cmap='jet_r', vmin=1500, vmax=4146.96)
ax.set_title('Inversion Process Visualization')
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Z Coordinate')
# Create a horizontal colorbar
cbar = fig.colorbar(cax, ax=ax, orientation='horizontal', fraction=0.046, pad=0.2)
cbar.set_label('Velocity (m/s)')
# Adjust the layout to minimize white space
plt.subplots_adjust(top=0.85, bottom=0.2, left=0.1, right=0.9)
# Initialization function
def init():
    cax.set_array(iter_vp[0])  # Use the 2D array directly
    return cax,
# Animation function
def animate(i):
    cax.set_array(iter_vp[i])  # Update with the i-th iteration directly
    return cax,
# Create the animation
ani = animation.FuncAnimation(fig, animate, init_func=init, frames=len(iter_vp), interval=200, blit=True)
# Save the animation as a video file (e.g., MP4 format)
ani.save(os.path.join(project_path,f"inversion-4layer-64-32-16-16/inversion_process.gif"), writer='pillow', fps=10)
# Display the animation using HTML
plt.close(fig)  # Prevents static display of the last frame
HTML(ani.to_jshtml())

## 5 Layer CNN

In [ ]:
###########################################
# visualize the inversion results
###########################################
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

iter_vp = np.load(os.path.join(project_path,"inversion-5layer-256-32-32-32-16/iter_vp.npz"))["data"][::3]

# Set up the figure for plotting
fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.imshow(iter_vp[0], aspect='equal', cmap='jet_r', vmin=1500, vmax=4146.96)
ax.set_title('Inversion Process Visualization')
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Z Coordinate')
# Create a horizontal colorbar
cbar = fig.colorbar(cax, ax=ax, orientation='horizontal', fraction=0.046, pad=0.2)
cbar.set_label('Velocity (m/s)')
# Adjust the layout to minimize white space
plt.subplots_adjust(top=0.85, bottom=0.2, left=0.1, right=0.9)
# Initialization function
def init():
    cax.set_array(iter_vp[0])  # Use the 2D array directly
    return cax,
# Animation function
def animate(i):
    cax.set_array(iter_vp[i])  # Update with the i-th iteration directly
    return cax,
# Create the animation
ani = animation.FuncAnimation(fig, animate, init_func=init, frames=len(iter_vp), interval=200, blit=True)
# Save the animation as a video file (e.g., MP4 format)
ani.save(os.path.join(project_path,f"inversion-5layer-256-32-32-32-16/inversion_process.gif"), writer='pillow', fps=10)
# Display the animation using HTML
plt.close(fig)  # Prevents static display of the last frame
HTML(ani.to_jshtml())